In [ ]:
import math
from typing import Any
import torch
from torch import nn
# 将multihead中维度改为numhidden不切分就是selfattention（为什么这个顺序讲（（（

# 为了并行运算用positionalenoding代替顺序操作
class PostionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len = 1000, **kwargs: Any) -> None:
        super(PostionalEncoding, self).__init__( **kwargs)
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1,max_len, num_hiddens))
        X = torch.arange(1,max_len,dtype = torch.float32).reshape(-1,1) / torch.pow(1000,torch.arange(0,num_hiddens, 2, dtype = torch.float32)/num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X += self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

# 还可以通过先行投射实现相对位置编码